In [2]:
import sys
from pathlib import Path

import torch

# notebook 위치가 notebooks/ 안이므로 repo root를 path에 추가
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)

ROOT: C:\Users\cecil\Projects\Time-Series-Library


In [3]:
from argparse import Namespace
from custom.models import CoTSFATimesNet

In [4]:
args = Namespace(
    # basic
    task_name="long_term_forecast",
    
    # forecasting
    seq_len=96,
    label_len=48,
    pred_len=96,
    
    # input/output dimensions
    enc_in=7,
    dec_in=7,
    c_out=7,
    
    # TimesNet parameters
    d_model=16,
    d_ff=32,
    e_layers=2,
    d_layers=1,
    n_heads=8,
    factor=3,
    top_k=5,
    num_kernels=6,
    moving_avg=25,
    dropout=0.1,
    embed="timeF",
    freq="h",
    activation="gelu",
)

In [5]:
model = CoTSFATimesNet(args)
model.eval()

print(model.__class__)

<class 'custom.models.timesnet_cotsfa.CoTSFATimesNet'>


In [6]:
B = 4
seq_len = 96
label_len = 48
pred_len = 96
C = 7

# ETTh1은 hourly data라서 time feature dimension이 보통 4개
time_feat_dim = 4

x_enc = torch.randn(B, seq_len, C)
x_enc_aug = x_enc + 0.05 * torch.randn(B, seq_len, C)

x_mark_enc = torch.randn(B, seq_len, time_feat_dim)

x_dec = torch.randn(B, label_len + pred_len, C)
x_mark_dec = torch.randn(B, label_len + pred_len, time_feat_dim)

print("x_enc:", x_enc.shape)
print("x_enc_aug:", x_enc_aug.shape)
print("x_mark_enc:", x_mark_enc.shape)
print("x_dec:", x_dec.shape)
print("x_mark_dec:", x_mark_dec.shape)

x_enc: torch.Size([4, 96, 7])
x_enc_aug: torch.Size([4, 96, 7])
x_mark_enc: torch.Size([4, 96, 4])
x_dec: torch.Size([4, 144, 7])
x_mark_dec: torch.Size([4, 144, 4])


In [7]:
with torch.no_grad():
    y_hat = model(x_enc, x_mark_enc, x_dec, x_mark_dec)

print("standard y_hat shape:", y_hat.shape)

standard y_hat shape: torch.Size([4, 96, 7])


In [8]:
with torch.no_grad():
    y_hat_latent, z = model.forward_with_latent(
        x_enc=x_enc,
        x_mark_enc=x_mark_enc,
        x_dec=x_dec,
        x_mark_dec=x_mark_dec,
        latent_slice="pred",
    )

print("y_hat_latent shape:", y_hat_latent.shape)
print("z shape:", z.shape)

y_hat_latent shape: torch.Size([4, 96, 7])
z shape: torch.Size([4, 96, 16])


In [9]:
with torch.no_grad():
    y_hat, y_hat_aug, z, z_aug = model.forward_cotsfa(
        x_enc=x_enc,
        x_enc_aug=x_enc_aug,
        x_mark_enc=x_mark_enc,
        x_dec=x_dec,
        x_mark_dec=x_mark_dec,
        latent_slice="pred",
    )

print("y_hat shape:", y_hat.shape)
print("y_hat_aug shape:", y_hat_aug.shape)
print("z shape:", z.shape)
print("z_aug shape:", z_aug.shape)

y_hat shape: torch.Size([4, 96, 7])
y_hat_aug shape: torch.Size([4, 96, 7])
z shape: torch.Size([4, 96, 16])
z_aug shape: torch.Size([4, 96, 16])


In [10]:
from custom.losses import cosine_similarity_batch

sim_z = cosine_similarity_batch(z, z_aug)

print("sim_z shape:", sim_z.shape)
print("sim_z mean:", sim_z.mean().item())
print("sim_z values:", sim_z)

sim_z shape: torch.Size([4])
sim_z mean: 0.9987784624099731
sim_z values: tensor([0.9989, 0.9987, 0.9989, 0.9987])


In [11]:
from custom.losses import CoTSFAAlignmentLoss

loss_fn = CoTSFAAlignmentLoss()

# input-only sanity check: y_aug = y
y = torch.randn(B, pred_len, C)
y_aug = y.clone()

loss_align, sim_latent_mean, sim_output_mean = loss_fn(
    z=z,
    z_aug=z_aug,
    y=y,
    y_aug=y_aug,
)

print("Input-only case")
print("loss_align:", loss_align.item())
print("sim_latent_mean:", sim_latent_mean.item())
print("sim_output_mean:", sim_output_mean.item())

Input-only case
loss_align: 0.001221492886543274
sim_latent_mean: 0.9987784624099731
sim_output_mean: 1.0


In [12]:
# input-output changed sanity check
y_aug_changed = y + 0.5 * torch.randn_like(y)

loss_align, sim_latent_mean, sim_output_mean = loss_fn(
    z=z,
    z_aug=z_aug,
    y=y,
    y_aug=y_aug_changed,
)

print("Input-output changed case")
print("loss_align:", loss_align.item())
print("sim_latent_mean:", sim_latent_mean.item())
print("sim_output_mean:", sim_output_mean.item())

Input-output changed case
loss_align: 0.10353676974773407
sim_latent_mean: 0.9987784624099731
sim_output_mean: 0.8952417373657227
